In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
import cv2
import os
import json
from PIL import Image, ImageDraw, ImageFont
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from PIL import Image

# Define constants
IMG_SIZE = 224  # Input size for the model

# Load the model
model_path = "MobileNetV2_224size_64bs_15E_all_trainable_255prepro.h5"  # Change to your .h5 model path if needed
model = tf.keras.models.load_model(model_path)

# Setting paths
base_path = 'nutritionverse-data'
coco_json_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images/_annotations.coco.json')

# Load COCO format metadata
with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# Create LabelEncoder
category_ids = [cat['id'] for cat in coco_data['categories']]
label_encoder = LabelEncoder()
label_encoder.fit(category_ids)

IMG_SIZE = 224  # Match your model's input size

def detect_food_in_image_pil(image: Image.Image) -> tuple:
    image = np.array(image.convert("RGB"))
    original_image = image.copy()
    results = []

    # Sliding window detection
    window_sizes = [(image.shape[1] // 2, image.shape[0] // 2)]
    step_size = min(image.shape[0], image.shape[1]) // 1
    sliding_window_results = []

    for win_w, win_h in window_sizes:
        if win_w < 50 or win_h < 50:
            continue

        for y in range(0, image.shape[0] - win_h, step_size):
            for x in range(0, image.shape[1] - win_w, step_size):
                window = original_image[y:y + win_h, x:x + win_w]
                window_resized = cv2.resize(window, (IMG_SIZE, IMG_SIZE))
                window_normalized = window_resized / 255.0
                prediction = model.predict(np.expand_dims(window_normalized, axis=0), verbose=0)
                predicted_class_idx = np.argmax(prediction)
                confidence = prediction[0][predicted_class_idx]

                if confidence < 0.8:
                    continue

                predicted_class = label_encoder.inverse_transform([predicted_class_idx])[0]
                category_name = next((cat['name'] for cat in coco_data['categories']
                                      if cat['id'] == predicted_class), "Unknown")

                sliding_window_results.append({
                    'class_idx': predicted_class_idx,
                    'category': category_name,
                    'confidence': float(confidence),
                    'bbox': [x, y, win_w, win_h]
                })

    # NMS
    sliding_window_results.sort(key=lambda x: x['confidence'], reverse=True)
    keep_results = []

    for i, current in enumerate(sliding_window_results):
        x1, y1, w1, h1 = current['bbox']
        x2, y2 = x1 + w1, y1 + h1
        keep_results.append(current)

        for j, other in enumerate(sliding_window_results):
            if i == j or other in keep_results:
                continue
            x3, y3, w2, h2 = other['bbox']
            x4, y4 = x3 + w2, y3 + h2
            x_overlap = max(0, min(x2, x4) - max(x1, x3))
            y_overlap = max(0, min(y2, y4) - max(y1, y3))
            intersection = x_overlap * y_overlap
            area1 = w1 * h1
            area2 = w2 * h2
            union = area1 + area2 - intersection
            iou = intersection / union if union > 0 else 0
            if iou > 0.8 and current['category'] == other['category']:
                continue

    # Draw on image using PIL
    annotated_image = Image.fromarray(original_image)
    draw = ImageDraw.Draw(annotated_image)
    font = ImageFont.load_default()

    for result in keep_results:
        x, y, w, h = result['bbox']
        draw.rectangle([(x, y), (x + w, y + h)], outline="blue", width=3)
        draw.text((x, y - 10), f"{result['category']} ({result['confidence']:.2f})", fill="blue", font=font)
        results.append(result)

    return annotated_image, results


# Gradio Interface
interface = gr.Interface(
    fn=detect_food_in_image_pil,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Annotated Image"),
        gr.JSON(label="Detection Results")
    ],
    title="Food Detection",
    description="Upload an image to detect foods using a custom trained CNN model."
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
